# DSP: Digital Spatial Profiling Analysis

## What This Notebook Does
- Loads DSP expression matrices and sample annotations.\n- Produces marker heatmaps and group comparisons.\n- Includes repeated plotting blocks for different subsets/groups.

## Inputs
- DSP expression matrix (counts/normalized) referenced in the notebook.\n- Sample annotation tables defining groups used in heatmaps.

## Outputs
- Heatmaps and summary figures saved by plotting calls.\n- Optional intermediate tables written to disk.

## How To Run
1. Load data and verify sample order/group labels.\n2. Run preprocessing/normalization sections (if present).\n3. Run heatmap sections for each desired subset.

## Notes
- Heatmap readability often depends on scaling (scale = 'row' vs none); keep it consistent within a figure set.\n- If group colors mismatch, verify the factor levels and annotation color mapping.

<!-- CODEX_NOTEBOOK_OVERVIEW_V1 -->


# Preprocess

In [79]:
data_Q3norm_origin=read.csv("Q3_norm.csv")

In [1]:
library(stringr )

In [82]:
data_sample=read.csv("sample_data.csv")

In [83]:
head(data_sample)

,X,slide.name,scan.name,panel,roi,segment,aoi,area,tags,FileVersion,⋯,SeqSetId,Raw,Trimmed,Stitched,Aligned,umiQ30,rtsQ30,DeduplicatedReads,NTC_ID,NTC
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,⋯,<chr>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<chr>,<int>
1,DSP-1001250001923-A-A02.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,001 CD4 CD8 CD20,CD4,CD4-aoi-001,3377.18,NA,0.2,⋯,A01288:49:HFG5FDSX3,463047,453739,452859,353443,0.9973,0.9970,23497,DSP-1001250001923-A-A01.dcc,397513
2,DSP-1001250001923-A-A03.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,001 CD4 CD8 CD20,CD20,CD20-aoi-001,4790.98,NA,0.2,⋯,A01288:49:HFG5FDSX3,491713,482462,481622,455110,0.9975,0.9973,29508,DSP-1001250001923-A-A01.dcc,397513
3,DSP-1001250001923-A-A04.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,001 CD4 CD8 CD20,CD8,CD8-aoi-001,1511.14,NA,0.2,⋯,A01288:49:HFG5FDSX3,174787,170323,170045,159945,0.9972,0.9971,10834,DSP-1001250001923-A-A01.dcc,397513
4,DSP-1001250001923-A-A05.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,002 CD4,CD4,CD4-aoi-001,1131.26,NA,0.2,⋯,A01288:49:HFG5FDSX3,82811,80770,80566,76012,0.9973,0.9968,5907,DSP-1001250001923-A-A01.dcc,397513
5,DSP-1001250001923-A-A06.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,003 CD4 CD8,CD4,CD4-aoi-001,8658.12,NA,0.2,⋯,A01288:49:HFG5FDSX3,1051849,1029481,1027617,953620,0.9975,0.9972,59353,DSP-1001250001923-A-A01.dcc,397513
6,DSP-1001250001923-A-A07.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,003 CD4 CD8,CD8,CD8-aoi-001,6035.38,NA,0.2,⋯,A01288:49:HFG5FDSX3,328969,290511,289905,273621,0.9973,0.9968,22111,DSP-1001250001923-A-A01.dcc,397513


In [84]:
F=function(x){
    return(str_replace_all(x,"-","\\."))
}

In [85]:
data_sample$sample_name=sapply(data_sample$X,F)

In [86]:
rownames(data_sample)=data_sample$sample_name
data_sample$X=NULL
data_sample$sample_name=NULL

In [88]:
F_group=function(x){
    str_list=strsplit(x," ")[[1]]
    if ("mature" %in% str_list){
        return("Mature")
    }else if("immature" %in% str_list){
        return("Immature")
    }else if("sparse" %in% str_list){
        return("Sparse")
    }else{
        return("No")
    }
}

In [89]:
data_sample$Group=sapply(data_sample$roi,F_group)

In [90]:
data_sample

,slide.name,scan.name,panel,roi,segment,aoi,area,tags,FileVersion,SoftwareVersion,⋯,Raw,Trimmed,Stitched,Aligned,umiQ30,rtsQ30,DeduplicatedReads,NTC_ID,NTC,Group
,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<lgl>,<dbl>,<chr>,⋯,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<int>,<chr>,<int>,<chr>
DSP.1001250001923.A.A02.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,001 CD4 CD8 CD20,CD4,CD4-aoi-001,3377.18,NA,0.2,2.3.3.10,⋯,463047,453739,452859,353443,0.9973,0.9970,23497,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A03.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,001 CD4 CD8 CD20,CD20,CD20-aoi-001,4790.98,NA,0.2,2.3.3.10,⋯,491713,482462,481622,455110,0.9975,0.9973,29508,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A04.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,001 CD4 CD8 CD20,CD8,CD8-aoi-001,1511.14,NA,0.2,2.3.3.10,⋯,174787,170323,170045,159945,0.9972,0.9971,10834,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A05.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,002 CD4,CD4,CD4-aoi-001,1131.26,NA,0.2,2.3.3.10,⋯,82811,80770,80566,76012,0.9973,0.9968,5907,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A06.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,003 CD4 CD8,CD4,CD4-aoi-001,8658.12,NA,0.2,2.3.3.10,⋯,1051849,1029481,1027617,953620,0.9975,0.9972,59353,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A07.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,003 CD4 CD8,CD8,CD8-aoi-001,6035.38,NA,0.2,2.3.3.10,⋯,328969,290511,289905,273621,0.9973,0.9968,22111,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A08.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,006 CD8,CD8,CD8-aoi-001,6987.42,NA,0.2,2.3.3.10,⋯,442626,433525,432727,406812,0.9975,0.9973,30554,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A09.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,008 CD8 CD20,CD20,CD20-aoi-001,3163.47,NA,0.2,2.3.3.10,⋯,257914,253058,252489,238738,0.9974,0.9969,18004,DSP-1001250001923-A-A01.dcc,397513,No
DSP.1001250001923.A.A10.dcc,14s25061-,14s25061-2,(v1.0) Human NGS Whole Transcriptome Atlas RNA,008 CD8 CD20,CD8,CD8-aoi-001,2929.45,NA,0.2,2.3.3.10,⋯,229549,199948,199628,188091,0.9976,0.9970,16346,DSP-1001250001923-A-A01.dcc,397513,No


In [91]:
F_roi=function(x){
    return(strsplit(x," ")[[1]][1])
}

In [92]:
data_sample$roi_label=sapply(data_sample$roi,F_roi)

In [94]:
data_sample$sample_name=paste0(data_sample$scan.name," | ",data_sample$roi_label," | ",data_sample$segment," | ",data_sample$Group)


In [96]:
data_Q3norm_all=data_Q3norm_origin

In [97]:
sample_Q3norm=colnames(data_Q3norm_all)[2:length(colnames(data_Q3norm_all))]

In [98]:
data_sample_127=data_sample[sample_Q3norm,]

In [99]:
write.csv(data_sample_127,file="sample_data_127.csv")

In [100]:
colnames(data_Q3norm_all)=c("Gene",data_sample_127$sample_name)

In [102]:
write.csv(data_Q3norm_all,file="Q3norm_127_rename.csv")

In [105]:
data_sample_CD4=data_sample_127[which(data_sample_127$segment=="CD4"),]
data_sample_CD8=data_sample_127[which(data_sample_127$segment=="CD8"),]
data_sample_CD20=data_sample_127[which(data_sample_127$segment=="CD20"),]

data_Q3norm_CD4=data_Q3norm_origin[,c("X",rownames(data_sample_CD4))]
data_Q3norm_CD8=data_Q3norm_origin[,c("X",rownames(data_sample_CD8))]
data_Q3norm_CD20=data_Q3norm_origin[,c("X",rownames(data_sample_CD20))]

In [106]:
write.csv(data_sample_CD4,file="sample_CD4.csv")
write.csv(data_Q3norm_CD4,file="Q3norm_CD4.csv")
write.csv(data_sample_CD8,file="sample_CD8.csv")
write.csv(data_Q3norm_CD8,file="Q3norm_CD8.csv")
write.csv(data_sample_CD20,file="sample_CD20.csv")
write.csv(data_Q3norm_CD20,file="Q3norm_CD20.csv")

# CD4 CD8 CD20 markergene 


In [ ]:
data_sample_CD4=read.csv("sample_CD4.csv")
data_Q3norm_CD4=read.csv("Q3norm_CD4.csv")
data_sample_CD8=read.csv("sample_CD8.csv")
data_Q3norm_CD8=read.csv("Q3norm_CD8.csv")
data_sample_CD20=read.csv("sample_CD20.csv")
data_Q3norm_CD20=read.csv("Q3norm_CD20.csv")

In [ ]:
#marker gene all
#load data
data_sample_CD4=read.csv("sample_CD4.csv")
data_Q3norm_CD4=read.csv("Q3norm_CD4.csv")
data_sample_CD8=read.csv("sample_CD8.csv")
data_Q3norm_CD8=read.csv("Q3norm_CD8.csv")
data_sample_CD20=read.csv("sample_CD20.csv")
data_Q3norm_CD20=read.csv("Q3norm_CD20.csv")

data_Q3norm_CD4$`X.1`=NULL
colnames(data_Q3norm_CD4)=c("Gene",data_sample_CD4$sample_name)
rownames(data_Q3norm_CD4)=data_Q3norm_CD4$Gene
data_Q3norm_CD4$Gene=NULL

data_Q3norm_CD8$`X.1`=NULL
colnames(data_Q3norm_CD8)=c("Gene",data_sample_CD8$sample_name)
rownames(data_Q3norm_CD8)=data_Q3norm_CD8$Gene
data_Q3norm_CD8$Gene=NULL

data_Q3norm_CD20$`X.1`=NULL
colnames(data_Q3norm_CD20)=c("Gene",data_sample_CD20$sample_name)
rownames(data_Q3norm_CD20)=data_Q3norm_CD20$Gene
data_Q3norm_CD20$Gene=NULL
data_Q3norm_merge=cbind(data_Q3norm_CD4,data_Q3norm_CD8,data_Q3norm_CD20)

In [110]:
data_marker=data_Q3norm_merge[c("BLNK","CD19","CD79A","CD79B","MS4A1","CD37","CD4","FOXP3","IL7R","FOS","CD40LG","CD8A","CD8B","GZMK","EOMES","ITM2C","CCL19","CCL21","CXCL13","CCR7","SELL","LAMP3","CXCR4","CD86","BCL6"),]
Group=c(rep("CD4",dim(data_Q3norm_CD4)[2]),rep("CD8",dim(data_Q3norm_CD8)[2]),rep("CD20",dim(data_Q3norm_CD20)[2]))
data_marker_matrix=as.matrix(data_marker)
data_marker_log=as.data.frame(log10(data_marker_matrix))######log transformation



In [114]:
annotation_col = data.frame(
  Group = factor(rep(c("CD4", "CD8", "CD20"), c(dim(data_Q3norm_CD4)[2],dim(data_Q3norm_CD8)[2],dim(data_Q3norm_CD20)[2])))
)
rownames(annotation_col) = colnames(data_marker_log)

annotation_row = data.frame(
  Marker = factor(rep(c("CD4", "CD8", "B Cell","TLS marker"), c(5,5,6,9)))
)
rownames(annotation_row) = rownames(data_marker_log)




library(pheatmap)
#Define colors


#Define group colors

ann_colors=list(Group=c(CD20="blue",CD4="#00447E",CD8="#F34800"))



#Draw heatmap

pheatmap(data_marker_log, scale = "row", #row-wise scaling 
         show_rownames=TRUE, #show row names (gene names)
         show_colnames=TRUE,
         cluster_cols=FALSE, #do not cluster columns
         cluster_rows=FALSE,
         treeheight_row = 1, #adjust row dendrogram height
         annotation_col=annotation_col,
         annotation_row=annotation_row,#add column grouping information to column annotations
         clustering_distance_rows = "correlation", #optimize clustering distance
         color = colorRampPalette(c("blue", "white", "red"))(100),
         annotation_colors=ann_colors, #assign colors to annotation blocks
         cutree_rows=3, #split all rows into three groups according to the clustering tree
         gaps_col=c(dim(data_Q3norm_CD4)[2],dim(data_Q3norm_CD4)[2]+dim(data_Q3norm_CD8)[2]), #set column gap positions
         gaps_row=c(5,10,16),
         cellheight=16,cellwidth=4, #size of each cell
         fontsize_col=6, #font size for column names
         fontsize_row = 6, #font size for row names
         fontsize_annotation=0.1,
         border=FALSE,
         main = "All", #main title
         filename="heatmap_all_markers.png")


# compare between group

In [115]:
library(dplyr)
library(ggplot2)
library(ggrepel)
makedirs <- function(dir_list){ 
  for (i_dir in dir_list){ 
    if (!dir.exists(i_dir) ){ 
      dir.create(i_dir, recursive = TRUE) 
    } 
  } 
}

makedirs("4_group_annova")
setwd("4_group_annova")

for (cell_type in c("CD4","CD8","CD20")){
data_sample=read.csv(paste0("../sample_",cell_type,".csv"),row.names = 1)
data_Q3norm=read.csv(paste0("../Q3norm_",cell_type,".csv"),row.names = 1)
rownames(data_Q3norm)=data_Q3norm$X
data_Q3norm$X=NULL
data_Q3norm_tranform=as.data.frame(log2(apply(as.matrix(t(data_Q3norm)),2,as.numeric)))
rownames(data_Q3norm_tranform)=colnames(data_Q3norm)
group=data_sample$Group
anova_test <- function(x){
  test <- data.frame(group, x)
  return(kruskal.test(test$x ~ test$group)$p.value)
}
anova_result <- apply(data_Q3norm_tranform,2,anova_test)
table(anova_result < 0.05)
data_out=as.data.frame(anova_result[anova_result < 0.05])
write.table(data_out,file=paste0(cell_type,"_anova_result.txt"),quote = FALSE,sep = "\t")
adjust_result <- p.adjust(anova_result, method='BH')
}


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




# CD8T TLS VS NoTLS

In [4]:
library(dplyr)
library(ggplot2)
library(ggrepel)
makedirs <- function(dir_list){ 
  for (i_dir in dir_list){ 
    if (!dir.exists(i_dir) ){ 
      dir.create(i_dir, recursive = TRUE) 
    } 
  } 
}

makedirs("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/CD8_TLS_NoTLS")
setwd("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/CD8_TLS_NoTLS")

cell_type="CD8"
data_sample=read.csv(paste0("../sample_",cell_type,".csv"),row.names = 1)
data_Q3norm=read.csv(paste0("../Q3norm_",cell_type,".csv"),row.names = 1)
rownames(data_Q3norm)=data_Q3norm$X
data_Q3norm$X=NULL
data_Q3norm_tranform=as.data.frame(log2(apply(as.matrix(t(data_Q3norm)),2,as.numeric)))
rownames(data_Q3norm_tranform)=colnames(data_Q3norm)
group=data_sample$Group



In [13]:
data_Q3norm_tranform$group=group
data_Q3norm_tranform_2=data_Q3norm_tranform[which(data_Q3norm_tranform$group %in% c("No","Mature")),]
group=data_Q3norm_tranform_2$group
data_Q3norm_tranform_2$group=NULL

In [26]:
FC <- function(x){
  test <- data.frame(group, x)
  grouped <- group_by(test,group)
  mean_group=summarise(grouped,avg_test=mean(x))
  print(mean_group)
  #print(mean_group$avg_test)
  log2fc=(log2(mean_group$avg_test[1]/mean_group$avg_test[2]))
  return(log2fc)
}

FC_result <- apply(data_Q3norm_tranform_2,2,FC)
#table(anova_result > 1 | anova_result < (-1))
foldchange_data=as.data.frame(FC_result)

# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     2.21
2 No         2.80
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     2.03
2 No         2.02
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.63
2 No         2.12
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.86
2 No         2.25
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.65
2 No         1.85
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     2.12
2 No         2.23
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.48
2 No         1.94
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.64
2 No         1.66
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.63
2 No         1.84
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.56
2 No         1.97
# A tibble: 2 × 2
  group  avg_test
  <chr>     <dbl>
1 Mature     1.61
2 No         1.77
# A tibble

In [20]:
t_test <- function(x){
  test <- data.frame(group, x)
  return(t.test(test$x ~ test$group)$p.value)
}
t_result <- apply(data_Q3norm_tranform_2,2,t_test)
p_data=as.data.frame(t_result)

In [27]:
data_volcano=cbind(foldchange_data,p_data)
colnames(data_volcano)=c("log2FC","Pvalue")
data_out=data_volcano[which(data_volcano$Pvalue<0.05),]
data_out=data_out[order(data_out$log2FC,decreasing = T),]

In [30]:
write.table(data_out,file = paste(cell_type,"_mature_VS_No_all.txt",sep = ""),sep = "\t",row.names = T,quote = FALSE)

In [ ]:
FC <- function(x){
  test <- data.frame(group, x)
  grouped <- group_by(test,group)
  mean_group=summarise(grouped,avg_test=mean(x))
  #print(mean_group)
  #print(mean_group$avg_test)
  log2fc=(log2(mean_group$avg_test[1]/mean_group$avg_test[2]))
  return(log2fc)
}

FC_result <- apply(data_Q3norm_tranform_2,2,FC)
#table(anova_result > 1 | anova_result < (-1))
foldchange_data=as.data.frame(anova_result)
#foldchange_data=as.data.frame(anova_result[anova_result > 1 | anova_result < (-1)])
#gene_data=rownames(as.data.frame(anova_result[anova_result > 1 | anova_result < (-1)]))
#gene_data
#data_select=data_final[,gene_data]

t_test <- function(x){
  test <- data.frame(group, x)
  return(t.test(test$x ~ test$group)$p.value)
}
t_result <- apply(data_Q3norm_tranform_2,2,t_test)
table(t_result < 0.05)
adjust_result <- p.adjust(t_result, method='BH')

paj=as.data.frame(adjust_result)
head(paj)
table(adjust_result < 0.05)
filter_gene=rownames(as.data.frame(sort(adjust_result[adjust_result < 0.1])))

data_volcano=cbind(foldchange_data,p,paj)
colnames(data_volcano)=c("log2FC","Pvalue","P_adjust")
#data_out=data_volcano[which(data_volcano$Pvalue<0.05),]
data_out=data_volcano
data_out=data_out[order(data_out$log2FC),]
write.table(data_out,file = paste(cell_type,"_mature_VS_No_all.txt",sep = ""),sep = "\t",row.names = T)

In [ ]:
library(dplyr)
library(ggplot2)
library(ggrepel)
makedirs <- function(dir_list){ 
  for (i_dir in dir_list){ 
    if (!dir.exists(i_dir) ){ 
      dir.create(i_dir, recursive = TRUE) 
    } 
  } 
}

makedirs("CD8_TLS_NoTLS")
setwd("CD8_TLS_NoTLS")

cell_type="CD8"
data_sample=read.csv(paste0("../sample_",cell_type,".csv"),row.names = 1)
data_Q3norm=read.csv(paste0("../Q3norm_",cell_type,".csv"),row.names = 1)
rownames(data_Q3norm)=data_Q3norm$X
data_Q3norm$X=NULL
data_Q3norm_tranform=as.data.frame(log2(apply(as.matrix(t(data_Q3norm)),2,as.numeric)))
rownames(data_Q3norm_tranform)=colnames(data_Q3norm)
group=data_sample$Group
anova_test <- function(x){
  test <- data.frame(group, x)
  return(kruskal.test(test$x ~ test$group)$p.value)
}
anova_result <- apply(data_Q3norm_tranform,2,anova_test)
table(anova_result < 0.05)
data_out=as.data.frame(anova_result[anova_result < 0.05])
write.table(data_out,file=paste0(cell_type,"_anova_result.txt"),quote = FALSE,sep = "\t")
adjust_result <- p.adjust(anova_result, method='BH')


# CD8

In [116]:
data_sample_CD8=read.csv("sample_CD8.csv")
data_Q3norm_CD8=read.csv("Q3norm_CD8.csv")

In [117]:
data_sample_CD8_No=data_sample_CD8[which(data_sample_CD8$Group=="No"),]
data_sample_CD8_Sparse=data_sample_CD8[which(data_sample_CD8$Group=="Sparse"),]
data_sample_CD8_Immature=data_sample_CD8[which(data_sample_CD8$Group=="Immature"),]
data_sample_CD8_Mature=data_sample_CD8[which(data_sample_CD8$Group=="Mature"),]
data_sample_CD8_resort=rbind(data_sample_CD8_No,data_sample_CD8_Sparse,data_sample_CD8_Immature,data_sample_CD8_Mature)

In [119]:
head(data_Q3norm_CD8)

,X.1,X,DSP.1001250001923.A.A04.dcc,DSP.1001250001923.A.A07.dcc,DSP.1001250001923.A.A08.dcc,DSP.1001250001923.A.A10.dcc,DSP.1001250001923.A.B02.dcc,DSP.1001250001923.A.B06.dcc,DSP.1001250001923.A.B09.dcc,DSP.1001250001923.A.C03.dcc,⋯,DSP.1001660004082.B.F01.dcc,DSP.1001660004082.B.F04.dcc,DSP.1001660004082.B.F07.dcc,DSP.1001660004082.B.F10.dcc,DSP.1001660004082.B.G01.dcc,DSP.1001660004082.B.G04.dcc,DSP.1001660004082.B.G07.dcc,DSP.1001660004082.B.G10.dcc,DSP.1001660004082.B.H01.dcc,DSP.1001660004082.B.H04.dcc
,<int>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,1,A2M,12.636915,4.212305,4.212305,4.212305,4.212305,8.424610,4.212305,2.808203,⋯,2.106152,4.212305,14.041016,3.008789,6.017578,4.212305,12.636915,10.530762,7.822852,8.424610
2,2,ACADM,4.212305,2.106152,2.106152,4.212305,4.212305,2.106152,4.212305,1.404102,⋯,2.106152,8.424610,1.404102,4.814063,4.814063,7.020508,2.106152,4.212305,3.610547,4.212305
3,3,ACADS,4.212305,6.318457,2.106152,4.212305,4.212305,2.106152,4.212305,2.808203,⋯,2.106152,4.212305,2.808203,4.212305,2.407031,5.616407,4.212305,4.212305,3.008789,4.212305
4,4,ACAT1,4.212305,6.318457,4.212305,4.212305,4.212305,2.106152,4.212305,1.404102,⋯,8.424610,4.212305,5.616407,3.008789,6.017578,1.404102,2.106152,2.106152,6.017578,4.212305
5,5,ACVRL1,4.212305,4.212305,2.106152,4.212305,4.212305,2.106152,4.212305,2.808203,⋯,2.106152,4.212305,4.212305,1.805274,1.203516,2.808203,2.106152,5.265381,3.610547,1.053076
6,6,PSEN1,4.212305,6.318457,4.212305,4.212305,4.212305,2.106152,4.212305,8.424610,⋯,4.212305,8.424610,2.808203,2.407031,4.814063,2.808203,2.106152,1.053076,4.212305,7.371534


In [120]:
data_Q3norm_CD8$`X.1`=NULL
colnames(data_Q3norm_CD8)=c("Gene",data_sample_CD8$sample_name)
rownames(data_Q3norm_CD8)=data_Q3norm_CD8$Gene
data_Q3norm_CD8$Gene=NULL

In [122]:
data_Q3norm_CD8_resort=data_Q3norm_CD8[,data_sample_CD8_resort$sample_name]

In [128]:
KLF2_gene_list=c("BTG1","JUND","UBB","GZMM","KLRG1","TUBB2A","BTG2","CST7","GZMK","EGR1","KLF2","GADD45B","S1PR1","CCR7","TCF7","IL7R")
#KLF2_gene_list=c("BTG1","KLF2","BTG2","CCR7","TCF7","IL7R","GZMK")
Cytotoxicity_gene_list=c("PDCD1","LAG3","ENTPD1","GZMA","PRF1","IFNG","GZMB","NKG7","GNLY")
data_marker=data_Q3norm_CD8_resort[c(KLF2_gene_list,Cytotoxicity_gene_list),]
Group=c(rep("No",dim(data_sample_CD8_No)[1]),rep("Sparse",dim(data_sample_CD8_Sparse)[1]),rep("Immature",dim(data_sample_CD8_Immature)[1]),rep("Mature",dim(data_sample_CD8_Mature)[1]))
data_marker_matrix=as.matrix(data_marker)
data_marker_log=as.data.frame(log10(data_marker_matrix))######log transformation



In [130]:
annotation_col = data.frame(
  Group = factor(rep(c("No", "Sparse", "Immature","Mature"), c(dim(data_sample_CD8_No)[1],dim(data_sample_CD8_Sparse)[1],dim(data_sample_CD8_Immature)[1],dim(data_sample_CD8_Mature)[1])))
)
rownames(annotation_col) = colnames(data_marker_log)

annotation_row = data.frame(
  Marker = factor(rep(c("KLF2_signature","Cytotoxicity_signature"), c(16,9)))
)
rownames(annotation_row) = rownames(data_marker_log)




library(pheatmap)
#Define colors


#Define group colors

ann_colors=list(Group=c(No="blue",Sparse="#00447E",Immature="#F34800",Mature="green"))



#Draw heatmap

pheatmap(data_marker_log, scale = "row", #row-wise scaling 
         show_rownames=TRUE, #show row names (gene names)
         show_colnames=TRUE,
         cluster_cols=FALSE, #do not cluster columns
         cluster_rows=FALSE,
         treeheight_row = 1, #adjust row dendrogram height
         annotation_col=annotation_col,
         annotation_row=annotation_row,#add column grouping information to column annotations
         clustering_distance_rows = "correlation", #optimize clustering distance
         color = colorRampPalette(c("blue", "white", "red"))(100),
         annotation_colors=ann_colors, #assign colors to annotation blocks
         cutree_rows=3, #split all rows into three groups according to the clustering tree
         gaps_col=c(dim(data_sample_CD8_No)[1],dim(data_sample_CD8_No)[1]+dim(data_sample_CD8_Sparse)[1],dim(data_sample_CD8_No)[1]+dim(data_sample_CD8_Sparse)[1]+dim(data_sample_CD8_Immature)[1]), #set column gap positions
         #gaps_row=c(5,10,16),
         cellheight=32,cellwidth=10, #size of each cell
         fontsize_col=6, #font size for column names
         fontsize_row = 6, #font size for row names
         fontsize_annotation=0.1,
         border=FALSE,
         main = "All", #main title
         filename="heatmap_KLF2_sig_markers.png")


In [146]:
library(dplyr)
library(ggplot2)
library(ggrepel)
makedirs <- function(dir_list){ 
  for (i_dir in dir_list){ 
    if (!dir.exists(i_dir) ){ 
      dir.create(i_dir, recursive = TRUE) 
    } 
  } 
}

makedirs("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/4_group_annova_CD8")
setwd("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/4_group_annova_CD8")


anova_test <- function(x){
  test <- data.frame(Group, x)
  return(kruskal.test(test$x ~ test$Group)$p.value)
}
data_marker_log_t=as.data.frame(t(data_marker_log))
anova_result <- apply(data_marker_log_t,2,anova_test)
table(anova_result < 0.05)
write.table(as.data.frame(anova_result),file=paste0("CD8_anova_result.txt"),quote = FALSE,sep = "\t")

data_out=as.data.frame(anova_result[anova_result < 0.05])
write.table(data_out,file=paste0("CD8_anova_result_0.05.txt"),quote = FALSE,sep = "\t")
adjust_result <- p.adjust(anova_result, method='BH')



FALSE  TRUE 
   24     1 

In [152]:
data_Q3norm_CD8_out=as.data.frame(log10(t(data_Q3norm_CD8_resort)))

In [154]:
write.csv(data_Q3norm_CD8_out,file="/NFS_home/NFS_home_2/zhouchi/ICC/DSP/CD8T_exp.csv")

In [36]:
df_CD8T=read.csv("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/DSP_CD8T_AOI_exp.csv",row.names = 1)
df_CD8T_meta=df_CD8T[,c("Group1","Group2")]
df_CD8T$Group1=NULL
df_CD8T$Group2=NULL
TLS_imprint=c("IGHA1", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHGP", "IGHM", "IGKC", "IGLC1", "IGLC2", "IGLC3", "JCHAIN", "CD52", "CD79A", "FCRL5", "MZB1", "SSR4", "XBP1", "TRBC2", "IL7R", "CXCL12", "LUM", "C1QA", "C7", "APOE", "PTLP","PTGDS", "PIM2", "DERL3")
TLS_imprint=setdiff(TLS_imprint,setdiff(TLS_imprint,colnames(df_CD8T)))
TLS_imprint_expr=df_CD8T[,TLS_imprint]
TLS_imprint_expr=na.omit(TLS_imprint_expr)
TLS_imprint_sig=as.data.frame(apply(TLS_imprint_expr,1,mean))
colnames(TLS_imprint_sig)="TLS_imprint"
df_CD8T$TLS_imprint=TLS_imprint_sig$TLS_imprint

KLF2_imprint_new=c("TCF7","CCR7","IL7R","KLRG1","GZMK","CST7","GZMM","SH2D1A","CRTAM")
KLF2_imprint_new=setdiff(KLF2_imprint_new,setdiff(KLF2_imprint_new,colnames(df_CD8T)))
KLF2_imprint_new_expr=df_CD8T[,KLF2_imprint_new]
KLF2_imprint_new_expr=na.omit(KLF2_imprint_new_expr)
KLF2_imprint_new_sig=as.data.frame(apply(KLF2_imprint_new_expr,1,mean))
colnames(KLF2_imprint_new_sig)="KLF2_imprint_new"
df_CD8T$KLF2_imprint_new=KLF2_imprint_new_sig$KLF2_imprint_new

library(msigdbr)
m_df = msigdbr(
  species = "Homo sapiens", 
  category = "H")
m_list = m_df %>% split(x = .$gene_symbol, f = .$gs_name)
#str(m_list)

Geneset1=m_list$HALLMARK_HYPOXIA
Geneset1 = unique(Geneset1) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset1=setdiff(Geneset1,setdiff(Geneset1,colnames(df_CD8T)))
Geneset1_expr=df_CD8T[,Geneset1]
Geneset1_expr=na.omit(Geneset1_expr)
Geneset1_sig=as.data.frame(apply(Geneset1_expr,1,mean))
colnames(Geneset1_sig)="Geneset1"
df_CD8T$Geneset1=Geneset1_sig$Geneset1

Geneset2=m_list$HALLMARK_APOPTOSIS
Geneset2 = unique(Geneset2) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset2=setdiff(Geneset2,setdiff(Geneset2,colnames(df_CD8T)))
Geneset2_expr=df_CD8T[,Geneset2]
Geneset2_expr=na.omit(Geneset2_expr)
Geneset2_sig=as.data.frame(apply(Geneset2_expr,1,mean))
colnames(Geneset2_sig)="Geneset2"
df_CD8T$Geneset2=Geneset2_sig$Geneset2

Geneset3=m_list$HALLMARK_GLYCOLYSIS
Geneset3 = unique(Geneset3) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset3=setdiff(Geneset3,setdiff(Geneset3,colnames(df_CD8T)))
Geneset3_expr=df_CD8T[,Geneset3]
Geneset3_expr=na.omit(Geneset3_expr)
Geneset3_sig=as.data.frame(apply(Geneset3_expr,1,mean))
colnames(Geneset3_sig)="Geneset3"
df_CD8T$Geneset3=Geneset3_sig$Geneset3

Geneset4=m_list$HALLMARK_IL2_STAT5_SIGNALING
Geneset4 = unique(Geneset4) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset4=setdiff(Geneset4,setdiff(Geneset4,colnames(df_CD8T)))
Geneset4_expr=df_CD8T[,Geneset4]
Geneset4_expr=na.omit(Geneset4_expr)
Geneset4_sig=as.data.frame(apply(Geneset4_expr,1,mean))
colnames(Geneset4_sig)="Geneset4"
df_CD8T$Geneset4=Geneset4_sig$Geneset4

Geneset5=m_list$HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY
Geneset5 = unique(Geneset5) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset5=setdiff(Geneset5,setdiff(Geneset5,colnames(df_CD8T)))
Geneset5_expr=df_CD8T[,Geneset5]
Geneset5_expr=na.omit(Geneset5_expr)
Geneset5_sig=as.data.frame(apply(Geneset5_expr,1,mean))
colnames(Geneset5_sig)="Geneset5"
df_CD8T$Geneset5=Geneset5_sig$Geneset5

Geneset6=m_list$HALLMARK_TGF_BETA_SIGNALING
Geneset6 = unique(Geneset6) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset6=setdiff(Geneset6,setdiff(Geneset6,colnames(df_CD8T)))
Geneset6_expr=df_CD8T[,Geneset6]
Geneset6_expr=na.omit(Geneset6_expr)
Geneset6_sig=as.data.frame(apply(Geneset6_expr,1,mean))
colnames(Geneset6_sig)="Geneset6"
df_CD8T$Geneset6=Geneset6_sig$Geneset6

Geneset7=m_list$HALLMARK_INFLAMMATORY_RESPONSE
Geneset7 = unique(Geneset7) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset7=setdiff(Geneset7,setdiff(Geneset7,colnames(df_CD8T)))
Geneset7_expr=df_CD8T[,Geneset7]
Geneset7_expr=na.omit(Geneset7_expr)
Geneset7_sig=as.data.frame(apply(Geneset7_expr,1,mean))
colnames(Geneset7_sig)="Geneset7"
df_CD8T$Geneset7=Geneset7_sig$Geneset7

Geneset8=m_list$HALLMARK_ALLOGRAFT_REJECTION
Geneset8 = unique(Geneset8) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset8=setdiff(Geneset8,setdiff(Geneset8,colnames(df_CD8T)))
Geneset8_expr=df_CD8T[,Geneset8]
Geneset8_expr=na.omit(Geneset8_expr)
Geneset8_sig=as.data.frame(apply(Geneset8_expr,1,mean))
colnames(Geneset8_sig)="Geneset8"
df_CD8T$Geneset8=Geneset8_sig$Geneset8

Geneset9=m_list$HALLMARK_TNFA_SIGNALING_VIA_NFKB
Geneset9 = unique(Geneset9) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset9=setdiff(Geneset9,setdiff(Geneset9,colnames(df_CD8T)))
Geneset9_expr=df_CD8T[,Geneset9]
Geneset9_expr=na.omit(Geneset9_expr)
Geneset9_sig=as.data.frame(apply(Geneset9_expr,1,mean))
colnames(Geneset9_sig)="Geneset9"
df_CD8T$Geneset9=Geneset9_sig$Geneset9

Geneset10=m_list$HALLMARK_MYOGENESIS
Geneset10 = unique(Geneset10) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset10=setdiff(Geneset10,setdiff(Geneset10,colnames(df_CD8T)))
Geneset10_expr=df_CD8T[,Geneset10]
Geneset10_expr=na.omit(Geneset10_expr)
Geneset10_sig=as.data.frame(apply(Geneset10_expr,1,mean))
colnames(Geneset10_sig)="Geneset10"
df_CD8T$Geneset10=Geneset10_sig$Geneset10


mm_df = msigdbr(
  species = "Homo sapiens", 
  category = "C2",subcategory = "CP:KEGG")
mm_listt = mm_df %>% split(x = .$gene_symbol, f = .$gs_name)
#str(m_list)
Geneset11=mm_listt$KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION#Ñ¡Ôñ»ùÒòÁÐ±í
Geneset11 = unique(Geneset11) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset11=setdiff(Geneset11,setdiff(Geneset11,colnames(df_CD8T)))
Geneset11_expr=df_CD8T[,Geneset11]
Geneset11_expr=na.omit(Geneset11_expr)
Geneset11_sig=as.data.frame(apply(Geneset11_expr,1,mean))
colnames(Geneset11_sig)="Geneset11"
df_CD8T$Geneset11=Geneset11_sig$Geneset11

df_CD8T_score=df_CD8T[,c("Geneset1","Geneset2","Geneset3",
                                             "Geneset4","Geneset5","Geneset6",
                                             "Geneset7","Geneset8","Geneset9",
                                             "Geneset10","Geneset11","TLS_imprint",
                                             "KLF2_imprint_new","IGHM","TCF7","IGHG1")]
df_CD8T_out=cbind(df_CD8T_meta,df_CD8T_score)


write.table(df_CD8T_out,file="/NFS_home/NFS_home_2/zhouchi/ICC/DSP/DSP_AOI_score.txt",sep="\t",quote = F)

# CD4

In [1]:
data_sample_CD4=read.csv("sample_CD4.csv")
data_Q3norm_CD4=read.csv("Q3norm_CD4.csv")

data_sample_CD4_No=data_sample_CD4[which(data_sample_CD4$Group=="No"),]
data_sample_CD4_Sparse=data_sample_CD4[which(data_sample_CD4$Group=="Sparse"),]
data_sample_CD4_Immature=data_sample_CD4[which(data_sample_CD4$Group=="Immature"),]
data_sample_CD4_Mature=data_sample_CD4[which(data_sample_CD4$Group=="Mature"),]
data_sample_CD4_resort=rbind(data_sample_CD4_No,data_sample_CD4_Sparse,data_sample_CD4_Immature,data_sample_CD4_Mature)

data_Q3norm_CD4$`X.1`=NULL
colnames(data_Q3norm_CD4)=c("Gene",data_sample_CD4$sample_name)
rownames(data_Q3norm_CD4)=data_Q3norm_CD4$Gene
data_Q3norm_CD4$Gene=NULL

data_Q3norm_CD4_resort=data_Q3norm_CD4[,data_sample_CD4_resort$sample_name]


data_Q3norm_CD4_out=as.data.frame(log10(t(data_Q3norm_CD4_resort)))


write.csv(data_Q3norm_CD4_out,file="/NFS_home/NFS_home_2/zhouchi/ICC/DSP/CD4T_exp.csv")

In [2]:
df_CD4T=read.csv("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/DSP_CD4T_AOI_exp.csv",row.names = 1)
df_CD4T_meta=df_CD4T[,c("Group1","Group2")]
df_CD4T$Group1=NULL
df_CD4T$Group2=NULL
TLS_imprint=c("IGHA1", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHGP", "IGHM", "IGKC", "IGLC1", "IGLC2", "IGLC3", "JCHAIN", "CD52", "CD79A", "FCRL5", "MZB1", "SSR4", "XBP1", "TRBC2", "IL7R", "CXCL12", "LUM", "C1QA", "C7", "APOE", "PTLP","PTGDS", "PIM2", "DERL3")
TLS_imprint=setdiff(TLS_imprint,setdiff(TLS_imprint,colnames(df_CD4T)))
TLS_imprint_expr=df_CD4T[,TLS_imprint]
TLS_imprint_expr=na.omit(TLS_imprint_expr)
TLS_imprint_sig=as.data.frame(apply(TLS_imprint_expr,1,mean))
colnames(TLS_imprint_sig)="TLS_imprint"
df_CD4T$TLS_imprint=TLS_imprint_sig$TLS_imprint

KLF2_imprint_new=c("TCF7","CCR7","IL7R","KLRG1","GZMK","CST7","GZMM","SH2D1A","CRTAM")
KLF2_imprint_new=setdiff(KLF2_imprint_new,setdiff(KLF2_imprint_new,colnames(df_CD4T)))
KLF2_imprint_new_expr=df_CD4T[,KLF2_imprint_new]
KLF2_imprint_new_expr=na.omit(KLF2_imprint_new_expr)
KLF2_imprint_new_sig=as.data.frame(apply(KLF2_imprint_new_expr,1,mean))
colnames(KLF2_imprint_new_sig)="KLF2_imprint_new"
df_CD4T$KLF2_imprint_new=KLF2_imprint_new_sig$KLF2_imprint_new

library(msigdbr)
m_df = msigdbr(
  species = "Homo sapiens", 
  category = "H")
m_list = m_df %>% split(x = .$gene_symbol, f = .$gs_name)
#str(m_list)

Geneset1=m_list$HALLMARK_HYPOXIA
Geneset1 = unique(Geneset1) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset1=setdiff(Geneset1,setdiff(Geneset1,colnames(df_CD4T)))
Geneset1_expr=df_CD4T[,Geneset1]
Geneset1_expr=na.omit(Geneset1_expr)
Geneset1_sig=as.data.frame(apply(Geneset1_expr,1,mean))
colnames(Geneset1_sig)="Geneset1"
df_CD4T$Geneset1=Geneset1_sig$Geneset1

Geneset2=m_list$HALLMARK_APOPTOSIS
Geneset2 = unique(Geneset2) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset2=setdiff(Geneset2,setdiff(Geneset2,colnames(df_CD4T)))
Geneset2_expr=df_CD4T[,Geneset2]
Geneset2_expr=na.omit(Geneset2_expr)
Geneset2_sig=as.data.frame(apply(Geneset2_expr,1,mean))
colnames(Geneset2_sig)="Geneset2"
df_CD4T$Geneset2=Geneset2_sig$Geneset2

Geneset3=m_list$HALLMARK_GLYCOLYSIS
Geneset3 = unique(Geneset3) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset3=setdiff(Geneset3,setdiff(Geneset3,colnames(df_CD4T)))
Geneset3_expr=df_CD4T[,Geneset3]
Geneset3_expr=na.omit(Geneset3_expr)
Geneset3_sig=as.data.frame(apply(Geneset3_expr,1,mean))
colnames(Geneset3_sig)="Geneset3"
df_CD4T$Geneset3=Geneset3_sig$Geneset3

Geneset4=m_list$HALLMARK_IL2_STAT5_SIGNALING
Geneset4 = unique(Geneset4) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset4=setdiff(Geneset4,setdiff(Geneset4,colnames(df_CD4T)))
Geneset4_expr=df_CD4T[,Geneset4]
Geneset4_expr=na.omit(Geneset4_expr)
Geneset4_sig=as.data.frame(apply(Geneset4_expr,1,mean))
colnames(Geneset4_sig)="Geneset4"
df_CD4T$Geneset4=Geneset4_sig$Geneset4

Geneset5=m_list$HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY
Geneset5 = unique(Geneset5) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset5=setdiff(Geneset5,setdiff(Geneset5,colnames(df_CD4T)))
Geneset5_expr=df_CD4T[,Geneset5]
Geneset5_expr=na.omit(Geneset5_expr)
Geneset5_sig=as.data.frame(apply(Geneset5_expr,1,mean))
colnames(Geneset5_sig)="Geneset5"
df_CD4T$Geneset5=Geneset5_sig$Geneset5

Geneset6=m_list$HALLMARK_TGF_BETA_SIGNALING
Geneset6 = unique(Geneset6) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset6=setdiff(Geneset6,setdiff(Geneset6,colnames(df_CD4T)))
Geneset6_expr=df_CD4T[,Geneset6]
Geneset6_expr=na.omit(Geneset6_expr)
Geneset6_sig=as.data.frame(apply(Geneset6_expr,1,mean))
colnames(Geneset6_sig)="Geneset6"
df_CD4T$Geneset6=Geneset6_sig$Geneset6

Geneset7=m_list$HALLMARK_INFLAMMATORY_RESPONSE
Geneset7 = unique(Geneset7) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset7=setdiff(Geneset7,setdiff(Geneset7,colnames(df_CD4T)))
Geneset7_expr=df_CD4T[,Geneset7]
Geneset7_expr=na.omit(Geneset7_expr)
Geneset7_sig=as.data.frame(apply(Geneset7_expr,1,mean))
colnames(Geneset7_sig)="Geneset7"
df_CD4T$Geneset7=Geneset7_sig$Geneset7

Geneset8=m_list$HALLMARK_ALLOGRAFT_REJECTION
Geneset8 = unique(Geneset8) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset8=setdiff(Geneset8,setdiff(Geneset8,colnames(df_CD4T)))
Geneset8_expr=df_CD4T[,Geneset8]
Geneset8_expr=na.omit(Geneset8_expr)
Geneset8_sig=as.data.frame(apply(Geneset8_expr,1,mean))
colnames(Geneset8_sig)="Geneset8"
df_CD4T$Geneset8=Geneset8_sig$Geneset8

Geneset9=m_list$HALLMARK_TNFA_SIGNALING_VIA_NFKB
Geneset9 = unique(Geneset9) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset9=setdiff(Geneset9,setdiff(Geneset9,colnames(df_CD4T)))
Geneset9_expr=df_CD4T[,Geneset9]
Geneset9_expr=na.omit(Geneset9_expr)
Geneset9_sig=as.data.frame(apply(Geneset9_expr,1,mean))
colnames(Geneset9_sig)="Geneset9"
df_CD4T$Geneset9=Geneset9_sig$Geneset9

Geneset10=m_list$HALLMARK_MYOGENESIS
Geneset10 = unique(Geneset10) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset10=setdiff(Geneset10,setdiff(Geneset10,colnames(df_CD4T)))
Geneset10_expr=df_CD4T[,Geneset10]
Geneset10_expr=na.omit(Geneset10_expr)
Geneset10_sig=as.data.frame(apply(Geneset10_expr,1,mean))
colnames(Geneset10_sig)="Geneset10"
df_CD4T$Geneset10=Geneset10_sig$Geneset10


mm_df = msigdbr(
  species = "Homo sapiens", 
  category = "C2",subcategory = "CP:KEGG")
mm_listt = mm_df %>% split(x = .$gene_symbol, f = .$gs_name)
#str(m_list)
Geneset11=mm_listt$KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION#Ñ¡Ôñ»ùÒòÁÐ±í
Geneset11 = unique(Geneset11) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset11=setdiff(Geneset11,setdiff(Geneset11,colnames(df_CD4T)))
Geneset11_expr=df_CD4T[,Geneset11]
Geneset11_expr=na.omit(Geneset11_expr)
Geneset11_sig=as.data.frame(apply(Geneset11_expr,1,mean))
colnames(Geneset11_sig)="Geneset11"
df_CD4T$Geneset11=Geneset11_sig$Geneset11

df_CD4T_score=df_CD4T[,c("Geneset1","Geneset2","Geneset3",
                                             "Geneset4","Geneset5","Geneset6",
                                             "Geneset7","Geneset8","Geneset9",
                                             "Geneset10","Geneset11","TLS_imprint",
                                             "KLF2_imprint_new","IGHM","TCF7","IGHG1")]
df_CD4T_out=cbind(df_CD4T_meta,df_CD4T_score)


write.table(df_CD4T_out,file="/NFS_home/NFS_home_2/zhouchi/ICC/DSP/DSP_AOI_score_CD4T.txt",sep="\t",quote = F)

# CD20

In [3]:
data_sample_CD20=read.csv("sample_CD20.csv")
data_Q3norm_CD20=read.csv("Q3norm_CD20.csv")

data_sample_CD20_No=data_sample_CD20[which(data_sample_CD20$Group=="No"),]
data_sample_CD20_Sparse=data_sample_CD20[which(data_sample_CD20$Group=="Sparse"),]
data_sample_CD20_Immature=data_sample_CD20[which(data_sample_CD20$Group=="Immature"),]
data_sample_CD20_Mature=data_sample_CD20[which(data_sample_CD20$Group=="Mature"),]
data_sample_CD20_resort=rbind(data_sample_CD20_No,data_sample_CD20_Sparse,data_sample_CD20_Immature,data_sample_CD20_Mature)

data_Q3norm_CD20$`X.1`=NULL
colnames(data_Q3norm_CD20)=c("Gene",data_sample_CD20$sample_name)
rownames(data_Q3norm_CD20)=data_Q3norm_CD20$Gene
data_Q3norm_CD20$Gene=NULL

data_Q3norm_CD20_resort=data_Q3norm_CD20[,data_sample_CD20_resort$sample_name]


data_Q3norm_CD20_out=as.data.frame(log10(t(data_Q3norm_CD20_resort)))


write.csv(data_Q3norm_CD20_out,file="/NFS_home/NFS_home_2/zhouchi/ICC/DSP/CD20_exp.csv")

In [4]:
df_CD20=read.csv("/NFS_home/NFS_home_2/zhouchi/ICC/DSP/DSP_CD20_AOI_exp.csv",row.names = 1)
df_CD20_meta=df_CD20[,c("Group1","Group2")]
df_CD20$Group1=NULL
df_CD20$Group2=NULL
TLS_imprint=c("IGHA1", "IGHG1", "IGHG2", "IGHG3", "IGHG4", "IGHGP", "IGHM", "IGKC", "IGLC1", "IGLC2", "IGLC3", "JCHAIN", "CD52", "CD79A", "FCRL5", "MZB1", "SSR4", "XBP1", "TRBC2", "IL7R", "CXCL12", "LUM", "C1QA", "C7", "APOE", "PTLP","PTGDS", "PIM2", "DERL3")
TLS_imprint=setdiff(TLS_imprint,setdiff(TLS_imprint,colnames(df_CD20)))
TLS_imprint_expr=df_CD20[,TLS_imprint]
TLS_imprint_expr=na.omit(TLS_imprint_expr)
TLS_imprint_sig=as.data.frame(apply(TLS_imprint_expr,1,mean))
colnames(TLS_imprint_sig)="TLS_imprint"
df_CD20$TLS_imprint=TLS_imprint_sig$TLS_imprint

KLF2_imprint_new=c("TCF7","CCR7","IL7R","KLRG1","GZMK","CST7","GZMM","SH2D1A","CRTAM")
KLF2_imprint_new=setdiff(KLF2_imprint_new,setdiff(KLF2_imprint_new,colnames(df_CD20)))
KLF2_imprint_new_expr=df_CD20[,KLF2_imprint_new]
KLF2_imprint_new_expr=na.omit(KLF2_imprint_new_expr)
KLF2_imprint_new_sig=as.data.frame(apply(KLF2_imprint_new_expr,1,mean))
colnames(KLF2_imprint_new_sig)="KLF2_imprint_new"
df_CD20$KLF2_imprint_new=KLF2_imprint_new_sig$KLF2_imprint_new

 

Geneset1=m_list$HALLMARK_HYPOXIA
Geneset1 = unique(Geneset1) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset1=setdiff(Geneset1,setdiff(Geneset1,colnames(df_CD20)))
Geneset1_expr=df_CD20[,Geneset1]
Geneset1_expr=na.omit(Geneset1_expr)
Geneset1_sig=as.data.frame(apply(Geneset1_expr,1,mean))
colnames(Geneset1_sig)="Geneset1"
df_CD20$Geneset1=Geneset1_sig$Geneset1

Geneset2=m_list$HALLMARK_APOPTOSIS
Geneset2 = unique(Geneset2) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset2=setdiff(Geneset2,setdiff(Geneset2,colnames(df_CD20)))
Geneset2_expr=df_CD20[,Geneset2]
Geneset2_expr=na.omit(Geneset2_expr)
Geneset2_sig=as.data.frame(apply(Geneset2_expr,1,mean))
colnames(Geneset2_sig)="Geneset2"
df_CD20$Geneset2=Geneset2_sig$Geneset2

Geneset3=m_list$HALLMARK_GLYCOLYSIS
Geneset3 = unique(Geneset3) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset3=setdiff(Geneset3,setdiff(Geneset3,colnames(df_CD20)))
Geneset3_expr=df_CD20[,Geneset3]
Geneset3_expr=na.omit(Geneset3_expr)
Geneset3_sig=as.data.frame(apply(Geneset3_expr,1,mean))
colnames(Geneset3_sig)="Geneset3"
df_CD20$Geneset3=Geneset3_sig$Geneset3

Geneset4=m_list$HALLMARK_IL2_STAT5_SIGNALING
Geneset4 = unique(Geneset4) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset4=setdiff(Geneset4,setdiff(Geneset4,colnames(df_CD20)))
Geneset4_expr=df_CD20[,Geneset4]
Geneset4_expr=na.omit(Geneset4_expr)
Geneset4_sig=as.data.frame(apply(Geneset4_expr,1,mean))
colnames(Geneset4_sig)="Geneset4"
df_CD20$Geneset4=Geneset4_sig$Geneset4

Geneset5=m_list$HALLMARK_REACTIVE_OXYGEN_SPECIES_PATHWAY
Geneset5 = unique(Geneset5) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset5=setdiff(Geneset5,setdiff(Geneset5,colnames(df_CD20)))
Geneset5_expr=df_CD20[,Geneset5]
Geneset5_expr=na.omit(Geneset5_expr)
Geneset5_sig=as.data.frame(apply(Geneset5_expr,1,mean))
colnames(Geneset5_sig)="Geneset5"
df_CD20$Geneset5=Geneset5_sig$Geneset5

Geneset6=m_list$HALLMARK_TGF_BETA_SIGNALING
Geneset6 = unique(Geneset6) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset6=setdiff(Geneset6,setdiff(Geneset6,colnames(df_CD20)))
Geneset6_expr=df_CD20[,Geneset6]
Geneset6_expr=na.omit(Geneset6_expr)
Geneset6_sig=as.data.frame(apply(Geneset6_expr,1,mean))
colnames(Geneset6_sig)="Geneset6"
df_CD20$Geneset6=Geneset6_sig$Geneset6

Geneset7=m_list$HALLMARK_INFLAMMATORY_RESPONSE
Geneset7 = unique(Geneset7) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset7=setdiff(Geneset7,setdiff(Geneset7,colnames(df_CD20)))
Geneset7_expr=df_CD20[,Geneset7]
Geneset7_expr=na.omit(Geneset7_expr)
Geneset7_sig=as.data.frame(apply(Geneset7_expr,1,mean))
colnames(Geneset7_sig)="Geneset7"
df_CD20$Geneset7=Geneset7_sig$Geneset7

Geneset8=m_list$HALLMARK_ALLOGRAFT_REJECTION
Geneset8 = unique(Geneset8) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset8=setdiff(Geneset8,setdiff(Geneset8,colnames(df_CD20)))
Geneset8_expr=df_CD20[,Geneset8]
Geneset8_expr=na.omit(Geneset8_expr)
Geneset8_sig=as.data.frame(apply(Geneset8_expr,1,mean))
colnames(Geneset8_sig)="Geneset8"
df_CD20$Geneset8=Geneset8_sig$Geneset8

Geneset9=m_list$HALLMARK_TNFA_SIGNALING_VIA_NFKB
Geneset9 = unique(Geneset9) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset9=setdiff(Geneset9,setdiff(Geneset9,colnames(df_CD20)))
Geneset9_expr=df_CD20[,Geneset9]
Geneset9_expr=na.omit(Geneset9_expr)
Geneset9_sig=as.data.frame(apply(Geneset9_expr,1,mean))
colnames(Geneset9_sig)="Geneset9"
df_CD20$Geneset9=Geneset9_sig$Geneset9

Geneset10=m_list$HALLMARK_MYOGENESIS
Geneset10 = unique(Geneset10) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset10=setdiff(Geneset10,setdiff(Geneset10,colnames(df_CD20)))
Geneset10_expr=df_CD20[,Geneset10]
Geneset10_expr=na.omit(Geneset10_expr)
Geneset10_sig=as.data.frame(apply(Geneset10_expr,1,mean))
colnames(Geneset10_sig)="Geneset10"
df_CD20$Geneset10=Geneset10_sig$Geneset10


mm_df = msigdbr(
  species = "Homo sapiens", 
  category = "C2",subcategory = "CP:KEGG")
mm_listt = mm_df %>% split(x = .$gene_symbol, f = .$gs_name)
#str(m_list)
Geneset11=mm_listt$KEGG_ANTIGEN_PROCESSING_AND_PRESENTATION#Ñ¡Ôñ»ùÒòÁÐ±í
Geneset11 = unique(Geneset11) #É¾È¥ÖØ¸´µÄ»ùÒòÃû
Geneset11=setdiff(Geneset11,setdiff(Geneset11,colnames(df_CD20)))
Geneset11_expr=df_CD20[,Geneset11]
Geneset11_expr=na.omit(Geneset11_expr)
Geneset11_sig=as.data.frame(apply(Geneset11_expr,1,mean))
colnames(Geneset11_sig)="Geneset11"
df_CD20$Geneset11=Geneset11_sig$Geneset11

df_CD20_score=df_CD20[,c("Geneset1","Geneset2","Geneset3",
                                             "Geneset4","Geneset5","Geneset6",
                                             "Geneset7","Geneset8","Geneset9",
                                             "Geneset10","Geneset11","TLS_imprint",
                                             "KLF2_imprint_new","IGHM","TCF7","IGHG1")]
df_CD20_out=cbind(df_CD20_meta,df_CD20_score)


write.table(df_CD20_out,file="/NFS_home/NFS_home_2/zhouchi/ICC/DSP/DSP_AOI_score_CD20.txt",sep="\t",quote = F)